<a href="https://colab.research.google.com/github/sjagadee/py_torch_from_basics/blob/main/07_nn_with_dataloaders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Breast Cancer Detection using a Simple Neural Network

In [1]:
! pip install torchinfo

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader

# Scikit-learn utilities for data preprocessing and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

### Import Libraries
This cell imports all the necessary libraries for data manipulation, machine learning, and neural network operations.

## Load and basic clean up

In [3]:
# Load the dataset from the provided URL
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")

# Display the first 5 rows to understand the structure
display(df.head())

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


### Load Data
This cell loads the breast cancer dataset from a URL into a pandas DataFrame and displays the first few rows.

In [4]:
df.shape

(569, 33)

### Check Data Shape
This cell shows the number of rows and columns in the DataFrame.

In [5]:
# Drop 'id' (not a feature) and 'Unnamed: 32' (contains NaN values)
df.drop(columns=["id", "Unnamed: 32"], inplace=True)

### Drop Unnecessary Columns
This cell removes the 'id' column, which is an identifier and not useful for training, and 'Unnamed: 32', which is an empty column.

In [6]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


### Display Cleaned Data
This cell displays the first few rows of the DataFrame after dropping the unnecessary columns.

## Train Test Split

In [7]:
# Separate features (X) and target label (y)
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# Split data into training (80%) and testing (20%) sets
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Separate Features and Target
This cell splits the DataFrame into features (`X`) and the target variable (`y`). It then performs a train-test split to create training and testing datasets.

In [8]:
# Standardize features to have zero mean and unit variance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Standardize Features
This cell uses `StandardScaler` to standardize the features in both the training and testing sets. Standardization is crucial for many machine learning algorithms, especially neural networks, to perform optimally.

In [9]:
X_train[:5]

array([[-1.44075296, -0.43531947, -1.36208497, -1.1391179 ,  0.78057331,
         0.71892128,  2.82313451, -0.11914956,  1.09266219,  2.45817261,
        -0.26380039, -0.01605246, -0.47041357, -0.47476088,  0.83836493,
         3.25102691,  8.43893667,  3.39198733,  2.62116574,  2.06120787,
        -1.23286131, -0.47630949, -1.24792009, -0.97396758,  0.72289445,
         1.18673232,  4.67282796,  0.9320124 ,  2.09724217,  1.88645014],
       [ 1.97409619,  1.73302577,  2.09167167,  1.85197292,  1.319843  ,
         3.42627493,  2.01311199,  2.66503199,  2.1270036 ,  1.55839569,
         0.80531919, -0.81268678,  0.75195659,  0.87716951, -0.89605315,
         1.18122247,  0.18362761,  0.60059598, -0.31771686,  0.52963649,
         2.17331385,  1.3112795 ,  2.08161691,  2.1374055 ,  0.76192793,
         3.26560084,  1.92862053,  2.6989469 ,  1.89116053,  2.49783848],
       [-1.39998202, -1.24962228, -1.34520926, -1.10978518, -1.33264483,
        -0.30735463, -0.36555756, -0.69650228,  1

### Display Scaled Training Data
This cell shows the first 5 rows of the standardized training data.

In [10]:
Y_train.head()

,diagnosis
68,B
181,M
63,B
248,B
60,B


### Display Target Training Data
This cell shows the first 5 target labels for the training data.

In [11]:
# Convert categorical labels ('M', 'B') into integers (1, 0)
encoder = LabelEncoder()
Y_train = encoder.fit_transform(Y_train)
Y_test = encoder.transform(Y_test)

### Encode Target Labels
This cell uses `LabelEncoder` to convert categorical target labels ('M' and 'B' for Malignant and Benign) into numerical format (0 and 1).

In [12]:
Y_train[:5]

array([0, 1, 0, 0, 0])

### Display Encoded Target Labels
This cell shows the first 5 encoded target labels for the training data.

## Numpy arrays to PyTorch tensors

In [13]:
# Convert NumPy arrays to PyTorch float tensors
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
Y_train_tensor = torch.from_numpy(Y_train.astype(np.float32))
Y_test_tensor = torch.from_numpy(Y_test.astype(np.float32))

### Convert to PyTorch Tensors
This cell converts the NumPy arrays of features and labels into PyTorch tensors. This is a necessary step before feeding the data into a PyTorch neural network model.

In [14]:
X_train_tensor.shape

torch.Size([455, 30])

### Display Shape of Feature Tensor
This cell displays the shape of the training features tensor, indicating the number of samples and features.

In [15]:
Y_train_tensor.shape

torch.Size([455])

### Display Shape of Target Tensor
This cell displays the shape of the training labels tensor.

## Let's now start with Neural Network

### Define the model

In [16]:
class MySimpleNN(nn.Module):
    """A simple single-layer neural network for binary classification."""

    def __init__(self, no_of_features):
        super().__init__()
        # Linear layer mapping features to a single logit
        self.linear = nn.Linear(no_of_features, 1)
        # Sigmoid activation to squash output between 0 and 1
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        """Forward pass logic."""
        output = self.linear(features)
        y_pred = self.sigmoid(output)
        return y_pred

### Model Initialization and Definition
This cell defines a simple neural network class `MySimpleNN`. It includes:
- `__init__`: Initializes the weights and bias of the single-layer neural network.
- `forward`: Implements the forward pass, calculating the output using a sigmoid activation function.
- `loss_function`: Calculates the binary cross-entropy loss, clamping predictions to avoid `log(0)`.

In [17]:
class BreastCancerDataset(Dataset):
    """Custom Dataset class to wrap features and labels for DataLoader usage."""
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        # Returns the total number of samples
        return len(self.features)

    def __getitem__(self, idx):
        # Returns one sample and its corresponding label
        return self.features[idx], self.labels[idx]

### Implementing mini-batch gradient descent

In [18]:
# Hyperparameters
learning_rate = 0.1
epochs = 30
mini_batch = 30

# Binary Cross Entropy Loss for classification
loss_function = nn.BCELoss()

# Initialize model with the number of input features (columns in X)
model = MySimpleNN(X_train_tensor.shape[1])

# Stochastic Gradient Descent optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Prepare PyTorch Datasets
dataset_train = BreastCancerDataset(X_train_tensor, Y_train_tensor)
dataset_test = BreastCancerDataset(X_test_tensor, Y_test_tensor)

# Prepare DataLoaders for batch processing
dataloader_train = DataLoader(dataset_train, batch_size=mini_batch, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=mini_batch, shuffle=True)

In [19]:
import time

# Set model to training mode
model.train()

start_time = time.time()

for epoch in range(epochs):
    epoch_loss = 0
    for batch_x, batch_y in dataloader_train:
        # 1. Forward pass: compute predicted outputs
        y_pred = model(batch_x)

        # 2. Calculate loss
        loss = loss_function(y_pred, batch_y.view(-1, 1))

        # 3. Clear existing gradients
        optimizer.zero_grad()

        # 4. Backward pass
        loss.backward()

        # 5. Optimization step
        optimizer.step()

        epoch_loss += loss.item()

    # Calculate average loss for the epoch
    avg_loss = epoch_loss / len(dataloader_train)

    # Log training progress with average loss and time elapsed
    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch + 1}/{epochs}] | Avg Loss: {avg_loss:.4f} | Time Elapsed: {elapsed:.2f}s")

Epoch [1/30] | Avg Loss: 0.2837 | Time Elapsed: 0.04s
Epoch [5/30] | Avg Loss: 0.1193 | Time Elapsed: 0.17s
Epoch [10/30] | Avg Loss: 0.0927 | Time Elapsed: 0.26s
Epoch [15/30] | Avg Loss: 0.0959 | Time Elapsed: 0.36s
Epoch [20/30] | Avg Loss: 0.0748 | Time Elapsed: 0.45s
Epoch [25/30] | Avg Loss: 0.0709 | Time Elapsed: 0.59s
Epoch [30/30] | Avg Loss: 0.0683 | Time Elapsed: 0.72s


In [20]:
# Set model to evaluation mode
model.eval()
accuracy_list = []

# Disable gradient calculation for testing
with torch.no_grad():
    for batch_x, batch_y in dataloader_test:
        y_pred = model(batch_x)

        # Apply threshold (0.5 or custom 0.8) to get binary classes
        y_pred_class = (y_pred > 0.8).float()

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred_class == batch_y.view(-1, 1)).float().mean()
        accuracy_list.append(batch_accuracy.item())

# Calculate and print the average accuracy across all test batches
accuracy = sum(accuracy_list) / len(accuracy_list)
print(f"Overall Test Accuracy: {accuracy * 100:.2f}%")

Overall Test Accuracy: 96.46%
